In [1]:
import re
from docx import Document
import pandas as pd
import numpy as np

# ================================
# Часть 1. Обработка файла с метками
# (Исходный код, который необходимо оставить)
# ================================

with open("./texts/entities_metka.txt", "r", encoding="utf-8") as infile:
    lines = infile.readlines()

entity_label_map = {}
for line in lines:
    line = line.strip()
    if not line:
        continue  # пропускаем пустые строки
    parts = line.rsplit(" ", maxsplit=1)
    if len(parts) != 2:
        continue  # если формат не соответствует, пропускаем строку
    phrase, label = parts
    if label == "VEH":
        label = "REW"
    entity_label_map[phrase] = label

# ================================
# Часть 2. Создание датасета из .docx файла с пропуском предложений без сущностей
# ================================

doc = Document("./texts/entities.docx")
data = []
sentence_counter = 1

# Предполагаем, что каждый параграф – отдельное предложение
for paragraph in doc.paragraphs:
    if not paragraph.text.strip():
        continue  # пропускаем пустые параграфы

    skip_sentence = False  # Флаг для пропуска предложения
    rows = []  # Список для хранения (токен, тег) предложения

    # Проходим по каждому run-у, чтобы сохранить информацию о форматировании
    for run in paragraph.runs:
        tokens = run.text.split()
        if run.bold:
            # Если текст выделен жирным, определяем всю фразу
            phrase = run.text.strip()
            # Если фраза не найдена в словаре, устанавливаем флаг для пропуска всего предложения
            if phrase not in entity_label_map:
                skip_sentence = True
                break
            label = entity_label_map[phrase]
            for idx, token in enumerate(tokens):
                tag = "B-" + label if idx == 0 else "I-" + label
                rows.append((token, tag))
        else:
            for token in tokens:
                rows.append((token, "O"))
                
    # Если флаг пропуска установлен или предложение пустое, пропускаем его
    if skip_sentence or not rows:
        continue

    # Формируем строки датасета: первая колонка — идентификатор предложения только для первого токена
    for i, (token, tag) in enumerate(rows):
        if i == 0:
            data.append((f"предложение {sentence_counter}", token, tag))
        else:
            data.append((np.nan, token, tag))
    sentence_counter += 1

# Создаем DataFrame и сохраняем в CSV
df = pd.DataFrame(data, columns=["Sentence", "Word", "Tag"])
df.to_csv("./texts/dataset.csv", index=False, encoding="utf-8-sig")

print("Датасет успешно создан и сохранен в ./texts/dataset.csv")


Датасет успешно создан и сохранен в ./texts/dataset.csv


In [2]:
data = pd.read_csv("./texts/dataset.csv")

In [3]:
data.head()

,Sentence,Word,Tag
0,предложение 1,Кутузов,B-PER
1,NaN,",",O
2,NaN,которого,O
3,NaN,он,O
4,NaN,догнал,O


In [4]:
data.Tag.value_counts()

Tag
O        6285
B-PER     235
I-PER      75
B-GPE      61
B-LOC       7
B-FAC       7
I-FAC       5
B-ORG       3
I-ORG       3
I-REW       2
I-LOC       1
B-REW       1
Name: count, dtype: int64

In [5]:
data.shape

(6685, 3)

In [6]:
len(data[data['Tag']=='O'])/len(data)

0.9401645474943904

In [7]:
from itertools import chain
def get_dict_map(data, token_or_tag):
    tok2idx = {}
    idx2tok = {}
    
    if token_or_tag == 'token':
        vocab = list(set(data['Word'].to_list()))
    else:
        vocab = list(set(data['Tag'].to_list()))
    
    idx2tok = {idx:tok for  idx, tok in enumerate(vocab)}
    tok2idx = {tok:idx for  idx, tok in enumerate(vocab)}
    return tok2idx, idx2tok


token2idx, idx2token = get_dict_map(data, 'token')
tag2idx, idx2tag = get_dict_map(data, 'tag')


In [8]:


# token2idx
tag2idx
idx2tag



{0: 'B-FAC',
 1: 'B-REW',
 2: 'B-GPE',
 3: 'B-PER',
 4: 'O',
 5: 'I-LOC',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'B-ORG',
 9: 'I-FAC',
 10: 'I-REW',
 11: 'I-PER'}

In [9]:
data['Word_idx'] = data['Word'].map(token2idx)
data['Tag_idx'] = data['Tag'].map(tag2idx)
data.head()

,Sentence,Word,Tag,Word_idx,Tag_idx
0,предложение 1,Кутузов,B-PER,1110,3
1,NaN,",",O,2091,4
2,NaN,которого,O,454,4
3,NaN,он,O,2508,4
4,NaN,догнал,O,2373,4


In [ ]:
data_fillna = data.fillna(method='ffill', axis=0)
# Группируем, передавая список столбцов
data_group = data_fillna.groupby(
    ['Sentence'], as_index=False
)[["Word", "Tag", "Word_idx", "Tag_idx"]].agg(lambda x: list(x))
# Смотрим результат
data_group.head()

/tmp/ipykernel_74480/2429894728.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_fillna = data.fillna(method='ffill', axis=0)


,Sentence,Word,Tag,Word_idx,Tag_idx
0,предложение 1,"[Кутузов, ,, которого, он, догнал, еще, в, Пол...","[B-PER, O, O, O, O, O, O, B-GPE, O, O, O, O, O...","[1110, 2091, 454, 2508, 2373, 2017, 616, 512, ...","[3, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, ..."
1,предложение 2,"[Глава, IX, Преследуемая, стотысячною, француз...","[O, O, O, O, O, O, O, O, B-PER, O, O, O, O, O,...","[1716, 978, 1417, 672, 2942, 1111, 1583, 923, ...","[4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, ..."
2,предложение 3,"[Глава, XIV, Кутузов, чрез, своего, лазутчика,...","[O, O, B-PER, O, O, O, O, O, O, O, O, O, O, O,...","[1716, 2022, 1110, 1956, 2623, 2052, 2747, 181...","[4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."
3,предложение 4,"[—, Покорно, благодарю,, я, теперь, один, прое...","[O, O, O, O, O, O, O, O, O, B-PER, I-PER, O, O...","[674, 2535, 3036, 2291, 1001, 2966, 1052, 674,...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 11, 4, 4, 4, 4,..."
4,предложение 5,"[Глава, XVI, Объехав, всю, линию, войск, от, п...","[O, O, O, O, O, O, O, O, O, O, O, B-PER, I-PER...","[1716, 900, 2522, 455, 2006, 3268, 892, 1101, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 11, 4, 4,..."


In [12]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("Все импорты выполнены успешно!")

2025-03-29 01:29:52.593125: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-29 01:29:52.593546: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 01:29:52.595814: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 01:29:52.601898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743200992.612242   74480 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743200992.61

Все импорты выполнены успешно!


In [13]:
def get_pad_train_test_val(data_group, data):

    #get max token and tag length
    n_token = len(list(set(data['Word'].to_list())))
    n_tag = len(list(set(data['Tag'].to_list())))

    #Pad tokens (X var)    
    tokens = data_group['Word_idx'].tolist()
    maxlen = max([len(s) for s in tokens])
    pad_tokens = pad_sequences(tokens, maxlen=maxlen, dtype='int32', padding='post', value= n_token - 1)

    #Pad Tags (y var) and convert it into one hot encoding
    tags = data_group['Tag_idx'].tolist()
    pad_tags = pad_sequences(tags, maxlen=maxlen, dtype='int32', padding='post', value= tag2idx["O"])
    n_tags = len(tag2idx)
    pad_tags = [to_categorical(i, num_classes=n_tags) for i in pad_tags]
    
    #Split train, test and validation set
    tokens_, test_tokens, tags_, test_tags = train_test_split(pad_tokens, pad_tags, test_size=0.1, train_size=0.9, random_state=2020)
    train_tokens, val_tokens, train_tags, val_tags = train_test_split(tokens_,tags_,test_size = 0.25,train_size =0.75, random_state=2020)

    print(
        'train_tokens length:', len(train_tokens),
        '\ntrain_tokens length:', len(train_tokens),
        '\ntest_tokens length:', len(test_tokens),
        '\ntest_tags:', len(test_tags),
        '\nval_tokens:', len(val_tokens),
        '\nval_tags:', len(val_tags),
    )
    
    return train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags

train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags = get_pad_train_test_val(data_group, data)

train_tokens length: 3 
train_tokens length: 3 
test_tokens length: 1 
test_tags: 1 
val_tokens: 1 
val_tags: 1


In [14]:
import numpy as np
import tensorflow
from tensorflow.keras import Sequential, Model, Input
from tensorflow.keras.layers import LSTM, Embedding, Dense, TimeDistributed, Dropout, Bidirectional
from tensorflow.keras.utils import plot_model

In [15]:
from numpy.random import seed
seed(1)
tensorflow.random.set_seed(2)

In [16]:
input_dim = len(list(set(data['Word'].to_list())))+1
output_dim = 64
input_length = max([len(s) for s in data_group['Word_idx'].tolist()])
n_tags = len(tag2idx)
print('input_dim: ', input_dim, '\noutput_dim: ', output_dim, '\ninput_length: ', input_length, '\nn_tags: ', n_tags)

input_dim:  3334 
output_dim:  64 
input_length:  2167 
n_tags:  12


In [17]:
def get_bilstm_lstm_model():
    model = Sequential()

    # Слой Embedding
    model.add(Embedding(input_dim=input_dim, output_dim=output_dim, input_length=input_length))

    # Слой bidirectional LSTM
    model.add(Bidirectional(LSTM(units=output_dim, return_sequences=True, dropout=0.2, recurrent_dropout=0.2), merge_mode = 'concat'))

    # Слой LSTM
    model.add(LSTM(units=output_dim, return_sequences=True, dropout=0.5, recurrent_dropout=0.5))

    # Слой timeDistributed Layer (обеспечивает выход формата many-to-many)
    model.add(TimeDistributed(Dense(n_tags, activation="relu")))

    #Optimiser 
    # adam = k.optimizers.Adam(lr=0.0005, beta_1=0.9, beta_2=0.999)

    # Compile model
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()
    
    return model

In [29]:
def train_model(X, y, model):
    loss = list()
    for i in range(3):
        # fit model for one epoch on this sequence
        hist = model.fit(X, y, batch_size=128, verbose=1, epochs=10, validation_split=0.2)
        loss.append(hist.history['loss'][0])
    return loss

In [30]:
results = pd.DataFrame()
model_bilstm_lstm = get_bilstm_lstm_model()

# Явно строим модель, указав размер входа
model_bilstm_lstm.build(input_shape=(None, input_length))

# Теперь можно визуализировать модель
plot_model(model_bilstm_lstm, show_shapes=True)

results['with_add_lstm'] = train_model(train_tokens, np.array(train_tags), model_bilstm_lstm)


/home/nika/Documents/Code/NER/.venv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

You must install pydot (`pip install pydot`) for `plot_model` to work.
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 55s 55s/step - accuracy: 0.2737 - loss: 9.3777 - val_accuracy: 0.9377 - val_loss: 0.9494
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6331 - loss: 1.4886 - val_accuracy: 0.9649 - val_loss: 0.6936
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9250 - loss: 0.9274 - val_accuracy: 0.9649 - val_loss: 0.6834
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9587 - loss: 0.8388 - val_accuracy: 0.9649 - val_loss: 0.6373
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6922 - loss: 0.9826 - val_accuracy: 0.9649 - val_loss: 0.6165
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9594 - loss: 0.6899 - val_accuracy: 0.9649 - val_loss: 0.5745
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9594 - loss: 0.6779 - val_accuracy: 0.9649 - val_loss: 0.5397
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9594 

In [22]:
predict = model_bilstm_lstm.predict(test_tokens)

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step


In [6]:
text = "В октябре 1805 года русские войска занимали села и города эрцгерцогства Австрийского, и еще новые полки приходили из России, и, отягощая постоем жителей, располагались у крепости Браунау."

text.find("Браунау")

179

In [ ]:
num = 100
np.argmax(predict[num], axis=1)

IndexError: index 1 is out of bounds for axis 0 with size 1

In [2]:
import docx

def extract_bold_phrases(docx_path, output_txt_path):
    # Открытие документа
    doc = docx.Document(docx_path)
    bold_phrases = []

    # Обрабатываем каждый параграф документа
    for para in doc.paragraphs:
        current_phrase = ""
        # Проходим по каждому фрагменту (run) в параграфе
        for run in para.runs:
            # Если данный run имеет установленное жирное начертание и содержит не пустой текст
            if run.bold and run.text.strip():
                # Если уже начат сбор фразы, добавляем пробел между словами, если необходимо
                if current_phrase:
                    # Если текущая фраза заканчивается пробелом, пробел добавлять не нужно
                    if not current_phrase[-1].isspace():
                        current_phrase += " " + run.text
                    else:
                        current_phrase += run.text
                else:
                    current_phrase = run.text
            else:
                # Если встретился не жирный текст, а мы собирали жирное слово/фразу, сохраняем результат
                if current_phrase:
                    bold_phrases.append(current_phrase.strip())
                    current_phrase = ""
        # Если в конце параграфа осталась собранная жирная фраза
        if current_phrase:
            bold_phrases.append(current_phrase.strip())

    # Запись собранных фраз в файл output_txt_path
    with open(output_txt_path, 'w', encoding='utf-8') as file:
        for phrase in bold_phrases:
            file.write(phrase + "\n")

if __name__ == "__main__":
    input_docx = "./texts/второй_файл.docx"   # Путь к вашему файлу .docx
    output_txt = "output.txt"   # Путь для создания файла .txt с результатом
    extract_bold_phrases(input_docx, output_txt)


In [3]:
import docx

def extract_unique_bold_phrases(docx_path, output_txt_path):
    # Открытие документа
    doc = docx.Document(docx_path)
    unique_bold_phrases = []
    seen_phrases = set()

    # Обрабатываем каждый абзац
    for para in doc.paragraphs:
        current_phrase = ""
        # Проходим по каждому run в абзаце
        for run in para.runs:
            if run.bold and run.text.strip():
                # Собираем смежные жирные фрагменты в одну строку
                if current_phrase:
                    # Добавляем пробел между словами, если это необходимо
                    if not current_phrase.endswith(" "):
                        current_phrase += " " + run.text
                    else:
                        current_phrase += run.text
                else:
                    current_phrase = run.text
            else:
                # Если встречается не жирный текст и уже есть собранная фраза
                if current_phrase:
                    phrase = current_phrase.strip()
                    if phrase not in seen_phrases:
                        unique_bold_phrases.append(phrase)
                        seen_phrases.add(phrase)
                    current_phrase = ""
        # Проверка после завершения абзаца, если осталась собранная жирная фраза
        if current_phrase:
            phrase = current_phrase.strip()
            if phrase not in seen_phrases:
                unique_bold_phrases.append(phrase)
                seen_phrases.add(phrase)

    # Запись уникальных фраз в файл output_txt_path
    with open(output_txt_path, 'w', encoding='utf-8') as file:
        for phrase in unique_bold_phrases:
            file.write(phrase + "\n")

if __name__ == "__main__":
    input_docx = "./texts/второй_файл.docx"   # Путь к исходному файлу .docx
    output_txt = "output.txt"   # Путь для файла с результатом
    extract_unique_bold_phrases(input_docx, output_txt)
